## Question 2 — Transition-Based Dependency Parser

This section implements a data-driven dependency parser using the Arc-Standard transition system and a classifier trained on the Universal Dependencies English-EWT treebank. The implementation includes CoNLL-U data processing, oracle simulation, POS-based feature extraction, classifier training, parsing, and evaluation using Labeled Attachment Score (LAS).


In [20]:
!pip install scikit-learn

### Download Universal Dependencies English-EWT

The Universal Dependencies English-EWT repository is cloned from GitHub to obtain the required CoNLL-U training and development datasets. The repository contains `en_ewt-ud-train.conllu`, `en_ewt-ud-dev.conllu`, and `en_ewt-ud-test.conllu` files used by the dependency parser.


In [21]:
!git clone https://github.com/UniversalDependencies/UD_English-EWT.git

fatal: destination path 'UD_English-EWT' already exists and is not an empty directory.


In [22]:
!ls UD_English-EWT

CONTRIBUTING.md        en_ewt-ud-train.conllu  not-to-release
en_ewt-ud-dev.conllu   eval.log		       README.md
en_ewt-ud-test.conllu  LICENSE.txt	       stats.xml


In [23]:
from collections import defaultdict

# ============================================
# PART 1A: READ CONLLU FILE
# ============================================

def read_conllu(filename):
    sentences = []
    sentence = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # End of sentence
            if line == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []
                continue

            # Ignore comments
            if line.startswith("#"):
                continue

            columns = line.split("\t")

            # CoNLL-U has 10 columns
            if len(columns) != 10:
                continue

            token_id = columns[0]

            # Ignore multi-word tokens such as 1-2
            if "-" in token_id or "." in token_id:
                continue

            token = {
                "id": int(columns[0]),
                "form": columns[1],
                "pos": columns[3],
                "head": int(columns[6]),
                "deprel": columns[7]
            }

            sentence.append(token)

    if sentence:
        sentences.append(sentence)

    return sentences


# Test reader
TRAIN_FILE = "UD_English-EWT/en_ewt-ud-train.conllu"

sentences = read_conllu(TRAIN_FILE)

print("Number of sentences:", len(sentences))

print("\nFirst sentence:\n")

for token in sentences[0]:
    print(
        token["id"],
        token["form"],
        token["pos"],
        token["head"],
        token["deprel"]
    )

Number of sentences: 12544

First sentence:

1 Al PROPN 0 root
2 - PUNCT 3 punct
3 Zaman PROPN 1 flat
4 : PUNCT 7 punct
5 American ADJ 6 amod
6 forces NOUN 7 nsubj
7 killed VERB 1 parataxis
8 Shaikh PROPN 7 obj
9 Abdullah PROPN 8 flat
10 al PROPN 8 flat
11 - PUNCT 12 punct
12 Ani PROPN 8 flat
13 , PUNCT 15 punct
14 the DET 15 det
15 preacher NOUN 8 appos
16 at ADP 18 case
17 the DET 18 det
18 mosque NOUN 15 nmod
19 in ADP 21 case
20 the DET 21 det
21 town NOUN 18 nmod
22 of ADP 23 case
23 Qaim PROPN 21 nmod
24 , PUNCT 28 punct
25 near ADP 28 case
26 the DET 28 det
27 Syrian ADJ 28 amod
28 border NOUN 21 nmod
29 . PUNCT 1 punct


In [24]:
# ============================================
# PART 1B: ORACLE
# ============================================

def has_unprocessed_child(node_id, stack, buffer, gold_children):

    remaining = set()

    for token in stack:
        remaining.add(token["id"])

    for token in buffer:
        remaining.add(token["id"])

    for child_id in gold_children[node_id]:
        if child_id in remaining:
            return True

    return False


def oracle(stack, buffer, gold_heads, gold_labels, gold_children):

    # Need at least two items for ARC
    if len(stack) >= 2:

        second = stack[-2]
        top = stack[-1]

        # --------------------------------
        # LEFT-ARC
        # TOP is HEAD of SECOND
        # --------------------------------

        if (
            second["id"] != 0
            and gold_heads[second["id"]] == top["id"]
        ):

            if not has_unprocessed_child(
                second["id"],
                stack,
                buffer,
                gold_children
            ):

                label = gold_labels[second["id"]]

                return "LEFT-ARC:" + label

        # --------------------------------
        # RIGHT-ARC
        # SECOND is HEAD of TOP
        # --------------------------------

        if gold_heads[top["id"]] == second["id"]:

            if not has_unprocessed_child(
                top["id"],
                stack,
                buffer,
                gold_children
            ):

                label = gold_labels[top["id"]]

                return "RIGHT-ARC:" + label

    # --------------------------------
    # SHIFT
    # --------------------------------

    if buffer:
        return "SHIFT"

    return None

In [25]:
# ============================================
# PART 1C: GENERATE TRAINING DATA
# ============================================

def generate_training_data(sentence):

    root = {
        "id": 0,
        "form": "ROOT",
        "pos": "ROOT",
        "head": -1,
        "deprel": "root"
    }

    stack = [root]
    buffer = sentence.copy()

    # Gold information
    gold_heads = {}
    gold_labels = {}
    gold_children = defaultdict(list)

    for token in sentence:

        token_id = token["id"]

        gold_heads[token_id] = token["head"]
        gold_labels[token_id] = token["deprel"]

        gold_children[token["head"]].append(token_id)

    training_examples = []

    # Simulate parsing
    while buffer or len(stack) > 1:

        configuration = {
            "stack": stack.copy(),
            "buffer": buffer.copy()
        }

        transition = oracle(
            stack,
            buffer,
            gold_heads,
            gold_labels,
            gold_children
        )

        if transition is None:
            break

        training_examples.append(
            (configuration, transition)
        )

        # -------------------------
        # APPLY TRANSITION
        # -------------------------

        if transition == "SHIFT":

            stack.append(buffer.pop(0))

        elif transition.startswith("LEFT-ARC:"):

            stack.pop(-2)

        elif transition.startswith("RIGHT-ARC:"):

            stack.pop()

    return training_examples

In [26]:
examples = generate_training_data(sentences[0])

print("Number of oracle examples:", len(examples))

print("\nFirst 10 oracle examples:\n")

for configuration, transition in examples[:10]:

    stack_words = [
        token["form"]
        for token in configuration["stack"]
    ]

    buffer_words = [
        token["form"]
        for token in configuration["buffer"]
    ]

    print("Stack:", stack_words)
    print("Buffer:", buffer_words)
    print("Transition:", transition)
    print("-" * 50)

Number of oracle examples: 58

First 10 oracle examples:

Stack: ['ROOT']
Buffer: ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Transition: SHIFT
--------------------------------------------------
Stack: ['ROOT', 'Al']
Buffer: ['-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Transition: SHIFT
--------------------------------------------------
Stack: ['ROOT', 'Al', '-']
Buffer: ['Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Transition: SHIFT
---------------------------------------

In [27]:
# ============================================
# PART 2A: FEATURE EXTRACTION
# ============================================

def get_pos(token):

    if token is None:
        return "<NULL>"

    return token["pos"]


def extract_features(configuration):

    stack = configuration["stack"]
    buffer = configuration["buffer"]

    # Top of stack
    if len(stack) >= 1:
        stack_top = stack[-1]
    else:
        stack_top = None

    # Second item of stack
    if len(stack) >= 2:
        stack_second = stack[-2]
    else:
        stack_second = None

    # First buffer item
    if len(buffer) >= 1:
        buffer_first = buffer[0]
    else:
        buffer_first = None

    # Second buffer item
    if len(buffer) >= 2:
        buffer_second = buffer[1]
    else:
        buffer_second = None

    return {
        "STACK_TOP_POS": get_pos(stack_top),
        "STACK_SECOND_POS": get_pos(stack_second),
        "BUFFER_FIRST_POS": get_pos(buffer_first),
        "BUFFER_SECOND_POS": get_pos(buffer_second)
    }

In [28]:
# ============================================
# PART 2B: TRAIN CLASSIFIER
# ============================================

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression


def train_model(training_sentences):

    X = []
    y = []

    print("Generating training data...")

    for i, sentence in enumerate(training_sentences):

        examples = generate_training_data(sentence)

        for configuration, transition in examples:

            features = extract_features(configuration)

            X.append(features)
            y.append(transition)

        if (i + 1) % 1000 == 0:
            print("Processed", i + 1, "sentences")

    print("\nTotal training examples:", len(X))

    print("\nTraining classifier...")

    vectorizer = DictVectorizer()

    X_vectorized = vectorizer.fit_transform(X)

    classifier = LogisticRegression(
        max_iter=1000
    )

    classifier.fit(
        X_vectorized,
        y
    )

    print("Training complete!")

    return vectorizer, classifier

In [29]:
# Train the dependency parser model

vectorizer, classifier = train_model(sentences)

print("vectorizer:", type(vectorizer))
print("classifier:", type(classifier))

Generating training data...
Processed 1000 sentences
Processed 2000 sentences
Processed 3000 sentences
Processed 4000 sentences
Processed 5000 sentences
Processed 6000 sentences
Processed 7000 sentences
Processed 8000 sentences
Processed 9000 sentences
Processed 10000 sentences
Processed 11000 sentences
Processed 12000 sentences

Total training examples: 407165

Training classifier...
Training complete!
vectorizer: <class 'sklearn.feature_extraction._dict_vectorizer.DictVectorizer'>
classifier: <class 'sklearn.linear_model._logistic.LogisticRegression'>


In [30]:
def apply_transition(stack, buffer, arcs, transition):

    if transition == "SHIFT":

        if buffer:
            stack.append(buffer.pop(0))

    elif transition.startswith("LEFT-ARC:"):

        label = transition.split(":", 1)[1]

        head = stack[-1]
        dependent = stack[-2]

        arcs.append(
            (head["id"], dependent["id"], label)
        )

        stack.pop(-2)

    elif transition.startswith("RIGHT-ARC:"):

        label = transition.split(":", 1)[1]

        head = stack[-2]
        dependent = stack[-1]

        arcs.append(
            (head["id"], dependent["id"], label)
        )

        stack.pop()

    return stack, buffer, arcs

In [31]:
def parse_sentence(sentence, vectorizer, classifier):

    root = {
        "id": 0,
        "form": "ROOT",
        "pos": "ROOT",
        "head": -1,
        "deprel": "root"
    }

    stack = [root]
    buffer = sentence.copy()

    arcs = []

    while buffer or len(stack) > 1:

        configuration = {
            "stack": stack,
            "buffer": buffer
        }

        features = extract_features(configuration)

        X = vectorizer.transform([features])

        # Get classifier probabilities
        probabilities = classifier.predict_proba(X)[0]
        classes = classifier.classes_

        # Select only legal transitions
        legal_transitions = []

        for transition in classes:

            if transition == "SHIFT":
                if buffer:
                    legal_transitions.append(transition)

            elif transition.startswith("LEFT-ARC:"):
                if len(stack) >= 2 and stack[-2]["id"] != 0:
                    legal_transitions.append(transition)

            elif transition.startswith("RIGHT-ARC:"):
                if len(stack) >= 2 and stack[-1]["id"] != 0:
                    legal_transitions.append(transition)

        # Choose the highest-probability legal transition
        best_transition = None
        best_probability = -1

        for transition, probability in zip(classes, probabilities):

            if transition in legal_transitions:
                if probability > best_probability:
                    best_probability = probability
                    best_transition = transition

        if best_transition is None:
            break

        stack, buffer, arcs = apply_transition(
            stack,
            buffer,
            arcs,
            best_transition
        )

    return arcs

In [32]:
test_sentence = sentences[0]

arcs = parse_sentence(
    test_sentence,
    vectorizer,
    classifier
)

print("Predicted dependency arcs:\n")

for head, dependent, label in arcs:

    head_word = "ROOT" if head == 0 else next(
        token["form"]
        for token in test_sentence
        if token["id"] == head
    )

    dependent_word = next(
        token["form"]
        for token in test_sentence
        if token["id"] == dependent
    )

    print(
        dependent_word,
        "-->",
        head_word,
        "(" + label + ")"
    )

Predicted dependency arcs:

- --> Zaman (punct)
Zaman --> Al (flat)
American --> forces (amod)
al --> Abdullah (flat)
Abdullah --> Shaikh (flat)
- --> Ani (punct)
Ani --> Shaikh (flat)
the --> preacher (det)
, --> preacher (punct)
Shaikh --> preacher (compound)
the --> mosque (det)
at --> mosque (case)
the --> town (det)
in --> town (case)
of --> Qaim (case)
Qaim --> town (nmod)
mosque --> town (compound)
preacher --> town (compound)
town --> killed (obj)
killed --> forces (acl)
: --> forces (punct)
Al --> forces (compound)
Syrian --> border (amod)
the --> border (det)
near --> border (case)
, --> border (punct)
border --> forces (nmod)
. --> forces (punct)
forces --> ROOT (root)


In [33]:
# Evaluate parser on the test set

TEST_FILE = "UD_English-EWT/en_ewt-ud-dev.conllu"

test_sentences = read_conllu(TEST_FILE)

print("Number of test sentences:", len(test_sentences))

correct = 0
total = 0

for i, sentence in enumerate(test_sentences):
    predicted_arcs = parse_sentence(sentence, vectorizer, classifier)

    # Convert predicted arcs to a set
    predicted_set = set(predicted_arcs)

    # Compare with gold dependencies
    for token in sentence:
        gold_arc = (
            token["head"],
            token["id"],
            token["deprel"]
        )

        if gold_arc in predicted_set:
            correct += 1

        total += 1

    if (i + 1) % 500 == 0:
        print("Evaluated", i + 1, "sentences")

las = (correct / total) * 100

print("\n==============================")
print("FINAL LAS:", round(las, 2), "%")
print("==============================")

Number of test sentences: 2001
Evaluated 500 sentences
Evaluated 1000 sentences
Evaluated 1500 sentences
Evaluated 2000 sentences

FINAL LAS: 56.71 %


In [34]:
# Show parser results for 3 test sentences

for i in range(3):
    sentence = test_sentences[i]

    predicted_arcs = parse_sentence(
        sentence,
        vectorizer,
        classifier
    )

    print("\n" + "=" * 60)
    print("Sentence", i + 1)
    print("=" * 60)

    print("Words:")
    print(" ".join(token["form"] for token in sentence))

    print("\nPredicted dependency arcs:")

    for head_id, dep_id, label in predicted_arcs:
        head_word = "ROOT" if head_id == 0 else sentence[head_id - 1]["form"]
        dep_word = sentence[dep_id - 1]["form"]

        print(
            f"{head_word} --> {dep_word} ({label})"
        )


Sentence 1
Words:
From the AP comes this story :

Predicted dependency arcs:
AP --> the (det)
AP --> From (case)
comes --> AP (nsubj)
story --> this (det)
comes --> story (obj)
comes --> : (punct)
ROOT --> comes (root)

Sentence 2
Words:
President Bush on Tuesday nominated two individuals to replace retiring jurists on federal courts in the Washington area .

Predicted dependency arcs:
Tuesday --> on (case)
Bush --> Tuesday (flat)
President --> Bush (flat)
nominated --> President (nsubj)
individuals --> two (nummod)
replace --> to (mark)
courts --> federal (amod)
courts --> on (case)
area --> Washington (compound)
area --> the (det)
area --> in (case)
courts --> area (nmod)
jurists --> courts (nmod)
retiring --> jurists (obj)
replace --> retiring (advcl)
individuals --> replace (acl)
nominated --> individuals (obj)
nominated --> . (punct)
ROOT --> nominated (root)

Sentence 3
Words:
Bush nominated Jennifer M. Anderson for a 15 - year term as associate judge of the Superior Court of th

In [35]:
# ============================================================
# REQUIRED EXAMPLE SENTENCES
# ============================================================

def make_test_sentence(words, pos_tags):

    sentence = []

    for i, (word, pos) in enumerate(
        zip(words, pos_tags),
        start=1
    ):
        sentence.append({
            "id": i,
            "form": word,
            "pos": pos,
            "head": -1,
            "deprel": "unknown"
        })

    return sentence


def print_arcs(sentence, arcs):

    word_map = {
        token["id"]: token["form"]
        for token in sentence
    }

    word_map[0] = "ROOT"

    for head, dependent, label in arcs:

        print(
            f"{word_map[head]} --> "
            f"{word_map[dependent]} "
            f"({label})"
        )


examples = [
    (
        ["The", "cat", "sat", "on", "the", "mat", "."],
        ["DET", "NOUN", "VERB", "ADP",
         "DET", "NOUN", "PUNCT"]
    ),

    (
        ["She", "eats", "a", "green", "salad", "."],
        ["PRON", "VERB", "DET", "ADJ",
         "NOUN", "PUNCT"]
    ),

    (
        ["I", "saw", "the", "man", "with",
         "a", "telescope", "."],
        ["PRON", "VERB", "DET", "NOUN",
         "ADP", "DET", "NOUN", "PUNCT"]
    )
]


for i, (words, pos_tags) in enumerate(examples, start=1):

    sentence = make_test_sentence(
        words,
        pos_tags
    )

    arcs = parse_sentence(
        sentence,
        vectorizer,
        classifier
    )

    print("\n" + "=" * 50)
    print("Example", i)
    print("=" * 50)

    print("Sentence:")
    print(" ".join(words))

    print("\nPredicted dependency arcs:")

    print_arcs(
        sentence,
        arcs
    )


Example 1
Sentence:
The cat sat on the mat .

Predicted dependency arcs:
cat --> The (det)
mat --> the (det)
mat --> on (case)
sat --> mat (obj)
cat --> sat (acl)
cat --> . (punct)
ROOT --> cat (root)

Example 2
Sentence:
She eats a green salad .

Predicted dependency arcs:
eats --> She (nsubj)
salad --> green (amod)
salad --> a (det)
eats --> salad (obj)
eats --> . (punct)
ROOT --> eats (root)

Example 3
Sentence:
I saw the man with a telescope .

Predicted dependency arcs:
saw --> I (nsubj)
man --> the (det)
telescope --> a (det)
telescope --> with (case)
man --> telescope (nmod)
saw --> man (obj)
saw --> . (punct)
ROOT --> saw (root)


QUESTION 2 – TRANSITION-BASED DEPENDENCY PARSER

Objective
The objective of this assignment is to implement and train a
data-driven transition-based dependency parser using the
Arc-Standard transition system.

Dataset
Training: en_ewt-ud-train.conllu
Evaluation: en_ewt-ud-dev.conllu

Part 1: Data Processing and Oracle Simulation
A CoNLL-U parser was implemented to read sentences and store
token ID, word form, POS tag, gold head, and dependency label.

An oracle simulator was implemented to generate training
instances. At each parser configuration, the oracle selects
SHIFT, LEFT-ARC(label), or RIGHT-ARC(label) according to the
gold dependency tree.

Part 2: Feature Extraction and Model Training
Four POS-based features were extracted:

1. POS tag of the top word on the stack
2. POS tag of the second word on the stack
3. POS tag of the first word in the buffer
4. POS tag of the second word in the buffer

A Logistic Regression classifier from scikit-learn was trained
using the generated configurations and oracle transitions.

Part 3: Parser and Evaluation
The parser maintains a stack, buffer, and dependency arcs.
At every step, the classifier predicts the next transition and
the corresponding transition is applied.

The parser was evaluated on the English-EWT development set.

Number of development sentences: 2001

Final Labeled Attachment Score (LAS): 57.17%

Example Sentences
The parser was tested on:

1. The cat sat on the mat.
2. She eats a green salad.
3. I saw the man with a telescope.

Conclusion
A complete Arc-Standard transition-based dependency parser was
implemented from scratch. The system performs CoNLL-U
processing, oracle simulation, feature extraction, classifier
training, dependency parsing, and LAS evaluation.

The final development-set LAS obtained was 56.71%